# LeetCode 368 — Largest Divisible Subset bằng Python

Link đề: https://leetcode.com/problems/largest-divisible-subset/ · Độ khó: **Medium** · Chủ đề: Array, Math, Dynamic Programming, Sorting

Skeleton LeetCode:

```python
class Solution:
    def largestDivisibleSubset(self, nums: List[int]) -> List[int]:
        ...
```

Ghi chú: đề và các ví dụ được viết theo trí nhớ về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code, nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ. Bài **cho phép nhiều đáp án đúng**, nên phần test dùng **hàm kiểm tra tính hợp lệ** thay vì so sánh từng phần tử.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: từ tập các số nguyên dương **đôi một khác nhau**, tìm **tập con lớn nhất** sao cho **mọi cặp** phần tử đều có số này **chia hết** số kia; nhiều đáp án thì trả **một** đáp án bất kỳ.
2. Nắm các công cụ Python: phép `%` kiểm tra chia hết, `sorted`/`sort`, vòng lặp lồng `for i in range(n): for j in range(i)`, mảng `dp`, mảng **`prev`** để dựng lại đáp án, `itertools.combinations`/bitmask để vét cạn, `functools.lru_cache`.
3. Đi từ cách **vét cạn mọi tập con** → **sắp xếp rồi quy hoạch động `O(n²)` kèm con trỏ `prev`** → **`dp[i]` lưu hẳn tập con** → **đệ quy có nhớ** (nhìn về phía lớn hơn).
4. Hiểu mấu chốt: tính **bắc cầu** của phép chia hết (`a | b`, `b | c` ⇒ `a | c`) cho phép chỉ kiểm tra **hai phần tử liền kề** trong dãy đã sắp xếp; vì vậy bài này **cùng khung với LIS** (Longest Increasing Subsequence).
5. Tránh các lỗi kinh điển: quên sắp xếp, tham lam sai, dựng lại đáp án không kiểm tra chia hết, quên `max` khi cập nhật `dp`, trả về độ dài thay vì tập con.

## 2. Đề bài & ví dụ

Cho một tập các số nguyên dương **khác nhau từng đôi một** `nums`. Hãy trả về **tập con lớn nhất** `answer` sao cho **mọi cặp** `(answer[i], answer[j])` trong tập con này thỏa:

- `answer[i] % answer[j] == 0`, **hoặc**
- `answer[j] % answer[i] == 0`.

Nếu có nhiều đáp án, trả về **một** đáp án bất kỳ.

**Ví dụ 1:** `nums = [1,2,3]` ⇒ `[1,2]` (đáp án `[1,3]` cũng được chấp nhận).

**Ví dụ 2:** `nums = [1,2,4,8]` ⇒ `[1,2,4,8]`.

**Ràng buộc:** `1 <= nums.length <= 1000`; `1 <= nums[i] <= 2 * 10^9`; mọi số trong `nums` **khác nhau**.

In [1]:
import itertools
import random
import time
from functools import lru_cache
from typing import List

examples = [
    ([1, 2, 3], 2),          # độ dài tối ưu (đáp án [1,2] hoặc [1,3])
    ([1, 2, 4, 8], 4),
]
for nums, best_len in examples:
    print(nums, "-> độ dài tối ưu", best_len)

[1, 2, 3] -> độ dài tối ưu 2
[1, 2, 4, 8] -> độ dài tối ưu 4


## 3. Kiến thức nền

### 3.1. Chia hết và phép `%`

`a % b == 0` nghĩa là **`a` chia hết cho `b`** (`b` là ước của `a`, ký hiệu `b | a`). Với **hai số dương khác nhau** `a < b`: số nhỏ **không thể** chia hết cho số lớn (vì `a % b = a ≠ 0`), nên điều kiện "một trong hai chia hết cho số kia" thực chất chỉ có thể là **số lớn chia hết cho số nhỏ**.

In [2]:
print(12 % 4, 4 % 12, 12 % 5)              # 0 4 2
print(8 % 2 == 0, 2 % 8 == 0)              # True False  (cặp (2, 8): số lớn chia hết số nhỏ)
a, b = 3, 4
print(a % b == 0 or b % a == 0)            # False: 3 và 4 không "chia hết" lẫn nhau

0 4 2
True False
False


### 3.2. Tính bắc cầu của chia hết

Nếu `a | b` và `b | c` thì `a | c`: viết `b = a·k₁`, `c = b·k₂ = a·(k₁k₂)`. Ví dụ `2 | 6` và `6 | 12` ⇒ `2 | 12`. Hệ quả cực kỳ quan trọng: một dãy `a₁ < a₂ < ... < a_k` mà **mỗi số chia hết số đứng ngay sau nó** (`a_i | a_{i+1}`) thì **mọi cặp** `(a_i, a_j)` với `i < j` đều có `a_i | a_j`. Ta gọi dãy đó là **chuỗi chia hết**. Vậy để kiểm tra "mọi cặp", **chỉ cần kiểm tra các cặp liền kề**.

In [3]:
chain = [2, 6, 12, 36]
print(all(chain[i + 1] % chain[i] == 0 for i in range(len(chain) - 1)))               # True: chỉ cặp liền kề
print(all(chain[j] % chain[i] == 0 for i in range(len(chain)) for j in range(i + 1, len(chain))))   # True: mọi cặp

True
True


### 3.3. Vì sao phải **sắp xếp**

Tập con cần "mọi cặp chia hết" **không phụ thuộc thứ tự**, nhưng khi ta duyệt mảng theo thứ tự cho trước, ta không biết ai "lớn chia cho nhỏ". Nếu **sắp xếp tăng dần**, với mọi `j < i` ta có `nums[j] < nums[i]`, nên điều kiện cặp `(j, i)` đơn giản là `nums[i] % nums[j] == 0`. Sắp xếp cũng cho phép ta nghĩ theo "dãy tăng dần" — giống LIS.

In [4]:
nums = [8, 1, 4, 2]
print(sorted(nums))                        # [1, 2, 4, 8]  (sorted tạo list mới, nums không đổi)
nums.sort()                                # sắp xếp tại chỗ (thay đổi nums)
print(nums)                                # [1, 2, 4, 8]

[1, 2, 4, 8]
[1, 2, 4, 8]


### 3.4. Vét cạn tập con: `itertools.combinations` và bitmask

Mảng `n` phần tử có `2^n` tập con. `itertools.combinations(nums, k)` sinh mọi tập con `k` phần tử. Cách khác: **bitmask** — số nguyên `mask` từ `0` đến `2^n − 1`, bit `i` bằng `1` nghĩa là chọn `nums[i]`. Hai cách đều chỉ dùng được khi `n` nhỏ (`<= 20`).

In [5]:
nums = [1, 2, 3]
print(list(itertools.combinations(nums, 2)))                  # [(1, 2), (1, 3), (2, 3)]
for mask in range(1 << len(nums)):                            # 1 << 3 = 8 tập con
    subset = [nums[i] for i in range(len(nums)) if mask >> i & 1]
    print(f"mask={mask:03b}: {subset}")

[(1, 2), (1, 3), (2, 3)]
mask=000: []
mask=001: [1]
mask=010: [2]
mask=011: [1, 2]
mask=100: [3]
mask=101: [1, 3]
mask=110: [2, 3]
mask=111: [1, 2, 3]


### 3.5. Mảng `dp`, hai vòng lặp lồng và mảng `prev`

Giống LIS, ta định nghĩa `dp[i]` = **độ dài chuỗi chia hết dài nhất kết thúc tại `nums[i]`** (sau khi sắp xếp). `dp[i] = 1 + max(dp[j])` trên các `j < i` có `nums[i] % nums[j] == 0` (hoặc `1` nếu không có `j` nào). Để **dựng lại** chuỗi, ta ghi thêm `prev[i]` = chỉ số `j` đã cho giá trị tốt nhất (hoặc `-1` nếu chuỗi bắt đầu tại `i`). Dựng lại: đi từ chỉ số của `max(dp)` ngược theo `prev` rồi **đảo** danh sách.

In [6]:
nums = [3, 4, 8, 16]                                          # đã sắp xếp
dp = [1] * len(nums)
prev = [-1] * len(nums)
for i in range(len(nums)):
    for j in range(i):
        if nums[i] % nums[j] == 0 and dp[j] + 1 > dp[i]:
            dp[i] = dp[j] + 1
            prev[i] = j
print("dp  :", dp)                                            # [1, 1, 2, 3]
print("prev:", prev)                                          # [-1, -1, 1, 2]

k = dp.index(max(dp))                                         # chỉ số kết thúc chuỗi dài nhất
chain = []
while k != -1:
    chain.append(nums[k])
    k = prev[k]
print(chain[::-1])                                            # [4, 8, 16]

dp  : [1, 1, 2, 3]
prev: [-1, -1, 1, 2]
[4, 8, 16]


### 3.6. Sao chép list và nối: `dp[j] + [x]`

`dp[j] + [x]` tạo **list mới** gồm các phần tử của `dp[j]` rồi thêm `x` (không sửa `dp[j]`). Ta sẽ dùng nó ở Cách 3 (mỗi `dp[i]` lưu hẳn chuỗi). Chú ý: `lst2 = lst1` **không** sao chép (hai tên cùng một list), còn `lst1 + []` hoặc `list(lst1)` mới sao chép.

In [7]:
a = [1, 2]
b = a + [4]                       # list mới
c = a                             # cùng một list!
c.append(99)
print(a, b, c)                    # [1, 2, 99] [1, 2, 4] [1, 2, 99]
print(max([[1], [1, 2], [1, 3]], key=len))      # [1, 2]  (max theo độ dài; chọn phần tử đầu tiên khi hòa)

[1, 2, 99] [1, 2, 4] [1, 2, 99]
[1, 2]


### 3.7. Đệ quy có nhớ: `functools.lru_cache`

`@lru_cache(maxsize=None)` đặt trên hàm đệ quy sẽ **nhớ** kết quả theo tham số, nên mỗi tham số chỉ tính **một lần**. Hữu ích khi nhiều nhánh đệ quy gọi lại cùng một trạng thái (ví dụ Fibonacci).

In [8]:
calls = 0


@lru_cache(maxsize=None)
def fib(n):
    global calls
    calls += 1
    return n if n < 2 else fib(n - 1) + fib(n - 2)


print(fib(30), "số lần thực sự tính:", calls)       # 832040, 31 (không có nhớ sẽ là hơn một triệu lần)

832040 số lần thực sự tính: 31


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

**Ví dụ A: `nums = [3, 4, 8, 16]`.** Với `3`: chưa có số nhỏ hơn ⇒ chuỗi `[3]` (độ dài `1`). Với `4`: `4 % 3 ≠ 0` ⇒ `[4]` (`1`). Với `8`: `8 % 4 == 0` (chuỗi `[4]` dài `1`) nên `[4, 8]` (`2`); `8 % 3 ≠ 0`. Với `16`: `16 % 8 == 0` (chuỗi `[4, 8]` dài `2`), `16 % 4 == 0` (dài `1`), nên ta nối vào chuỗi dài nhất ⇒ `[4, 8, 16]` (`3`).

**Ví dụ B (nhiều đáp án): `nums = [1, 2, 3, 6, 12]`.** `1` ⇒ `1`; `2` (nối `1`) ⇒ `2`; `3` (nối `1`) ⇒ `2`; `6` có thể nối vào chuỗi của `2` (`[1,2]`) hoặc của `3` (`[1,3]`), đều dài `2` ⇒ `3`; `12` nối vào chuỗi của `6` ⇒ `4`. Hai đáp án `[1,2,6,12]` và `[1,3,6,12]` cùng độ dài `4`.

| `i` | `nums[i]` | các `j` thỏa `nums[i] % nums[j] == 0` | `dp[i]` | `prev[i]` |
|---|---|---|---|---|
| 0 | 1 | – | 1 | −1 |
| 1 | 2 | `j=0 (1)` | 2 | 0 |
| 2 | 3 | `j=0 (1)` | 2 | 0 |
| 3 | 6 | `j=0, 1, 2` | 3 | 1 (hoặc 2) |
| 4 | 12 | `j=0, 1, 2, 3` | 4 | 3 |

Nhận xét: để đạt `dp[i]`, ta chọn `j` có `dp[j]` **lớn nhất** trong các `j` chia được `nums[i]` (không phải `j` gần nhất hay nhỏ nhất).

In [9]:
def divisible_pairs(nums):
    """In các cặp (nhỏ, lớn) thỏa lớn % nhỏ == 0 trong list đã sắp xếp."""
    nums = sorted(nums)
    return [(nums[j], nums[i]) for i in range(len(nums)) for j in range(i) if nums[i] % nums[j] == 0]


print(divisible_pairs([1, 2, 3, 6, 12]))
# [(1, 2), (1, 3), (1, 6), (2, 6), (3, 6), (1, 12), (2, 12), (3, 12), (6, 12)]

[(1, 2), (1, 3), (1, 6), (2, 6), (3, 6), (1, 12), (2, 12), (3, 12), (6, 12)]


## 5. Cách 1 — Brute Force / baseline: vét cạn mọi tập con

**Ý tưởng.** Thử **mọi tập con** (bitmask), kiểm tra **mọi cặp** có chia hết lẫn nhau không, giữ tập hợp lệ **lớn nhất**. Đúng theo định nghĩa, không cần mẹo nào, nhưng chỉ dùng được khi `n` nhỏ.

In [10]:
def is_divisible_set(items):
    """Mọi cặp (x, y) đều thỏa x % y == 0 hoặc y % x == 0."""
    for a, b in itertools.combinations(items, 2):
        if a % b != 0 and b % a != 0:
            return False
    return True


def largest_divisible_subset_brute(nums):
    n = len(nums)
    best = []
    for mask in range(1, 1 << n):
        subset = [nums[i] for i in range(n) if mask >> i & 1]
        if len(subset) > len(best) and is_divisible_set(subset):
            best = subset
    return best


for nums, best_len in examples:
    res = largest_divisible_subset_brute(nums)
    print(res, "độ dài", len(res), "(kỳ vọng", best_len, ")")

[1, 2] độ dài 2 (kỳ vọng 2 )
[1, 2, 4, 8] độ dài 4 (kỳ vọng 4 )


**Độ phức tạp (Cách 1).** Có `2^n` tập con, mỗi tập kiểm tra tối đa `O(n²)` cặp ⇒ **Time `O(2^n · n²)`**, **Space `O(n)`**. Với `n = 1000` là không thể (`2^1000`).

**Nút thắt:** ta kiểm tra lặp đi lặp lại những cặp giống nhau trong rất nhiều tập con, và **không dùng** tính bắc cầu. Nếu ta sắp xếp và coi tập hợp lệ là một **chuỗi** thì có thể xây dần từ chuỗi ngắn hơn.

## 6. Ý tưởng thuật toán / bước đột phá

1. **Sắp xếp** tăng dần. Khi đó mọi cặp `(nhỏ, lớn)` cần `lớn % nhỏ == 0`.
2. Nhờ **bắc cầu**, một tập hợp lệ khi và chỉ khi **dãy tăng dần** của nó có mỗi phần tử chia hết phần tử kế tiếp. Chứng minh ngắn: (⇒) hiển nhiên (liền kề là một cặp). (⇐) bắc cầu như mục 3.2.
3. Vậy ta cần **chuỗi chia hết dài nhất** — đúng khung **LIS**: `dp[i]` = độ dài chuỗi dài nhất **kết thúc** tại `nums[i]`; để kéo dài chuỗi kết thúc tại `nums[j]` bằng `nums[i]` ta chỉ cần `nums[i] % nums[j] == 0` (nhờ bắc cầu, mọi phần tử trước đó của chuỗi cũng chia được `nums[i]`).
4. `dp[i] = 1 + max(dp[j] : j < i, nums[i] % nums[j] == 0)`, mặc định `1`. Đáp án có độ dài `max(dp)`; dựng lại bằng `prev`.

Độ phức tạp `O(n²)` cho `n <= 1000` là `~500 000` phép so sánh — nhẹ.

In [11]:
nums = sorted([1, 2, 3, 6, 12])
dp = []
for i, x in enumerate(nums):
    candidates = [dp[j] for j in range(i) if x % nums[j] == 0]    # dp của các số nhỏ hơn chia được x
    dp.append(1 + max(candidates, default=0))
    print(f"x={x}: các dp[j] thỏa = {candidates} -> dp = {dp[-1]}")
print(dp, "độ dài tối ưu =", max(dp))

x=1: các dp[j] thỏa = [] -> dp = 1
x=2: các dp[j] thỏa = [1] -> dp = 2
x=3: các dp[j] thỏa = [1] -> dp = 2
x=6: các dp[j] thỏa = [1, 2, 2] -> dp = 3
x=12: các dp[j] thỏa = [1, 2, 2, 3] -> dp = 4
[1, 2, 2, 3, 4] độ dài tối ưu = 4


## 7. Các cách tiếp theo

### Cách 2 — Sắp xếp + DP `O(n²)` + con trỏ `prev` (lời giải chuẩn)

**Giải thích từng đoạn.**

1. `nums.sort()` (hoặc `sorted`): để lớn chia nhỏ.
2. `dp = [1] * n`, `prev = [-1] * n`: mỗi số tự nó là chuỗi độ dài `1`, chưa có tiền nhiệm.
3. Hai vòng lặp: với mỗi `i`, duyệt `j < i`; nếu `nums[i] % nums[j] == 0` **và** `dp[j] + 1 > dp[i]` thì cập nhật `dp[i]`, `prev[i] = j` (chọn `j` **tốt nhất**, không phải `j` cuối).
4. Theo dõi `best` = chỉ số có `dp` lớn nhất.
5. Dựng lại: từ `k = best` đi ngược `prev` đến `-1`, `append(nums[k])`; cuối cùng đảo ngược (hoặc không cần: mọi thứ tự đều được chấp nhận vì đề không yêu cầu thứ tự, nhưng ta đảo để ra dãy tăng cho dễ nhìn).

In [12]:
def largest_divisible_subset_dp(nums):
    nums = sorted(nums)                              # sorted: không làm thay đổi list của người gọi
    n = len(nums)
    dp = [1] * n                                     # dp[i]: độ dài chuỗi dài nhất kết thúc tại nums[i]
    prev = [-1] * n                                  # prev[i]: phần tử đứng ngay trước nums[i] trong chuỗi
    best = 0                                         # chỉ số kết thúc chuỗi dài nhất
    for i in range(n):
        for j in range(i):
            if nums[i] % nums[j] == 0 and dp[j] + 1 > dp[i]:
                dp[i] = dp[j] + 1
                prev[i] = j
        if dp[i] > dp[best]:
            best = i
    ans = []
    k = best
    while k != -1:                                   # đi ngược theo prev
        ans.append(nums[k])
        k = prev[k]
    return ans[::-1]


for nums, best_len in examples:
    res = largest_divisible_subset_dp(nums)
    print(res, "độ dài", len(res), "(kỳ vọng", best_len, ")")

[1, 2] độ dài 2 (kỳ vọng 2 )
[1, 2, 4, 8] độ dài 4 (kỳ vọng 4 )


In [13]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionDP:
    def largestDivisibleSubset(self, nums: List[int]) -> List[int]:
        nums.sort()
        n = len(nums)
        dp = [1] * n
        prev = [-1] * n
        best = 0
        for i in range(n):
            for j in range(i):
                if nums[i] % nums[j] == 0 and dp[j] + 1 > dp[i]:
                    dp[i] = dp[j] + 1
                    prev[i] = j
            if dp[i] > dp[best]:
                best = i
        ans = []
        while best != -1:
            ans.append(nums[best])
            best = prev[best]
        return ans[::-1]


print(SolutionDP().largestDivisibleSubset([1, 2, 3]), SolutionDP().largestDivisibleSubset([1, 2, 4, 8]))

[1, 2] [1, 2, 4, 8]


**Độ phức tạp Cách 2.** **Time `O(n²)`** (hai vòng lặp lồng; sắp xếp `O(n log n)` nhỏ hơn), **Space `O(n)`** (`dp`, `prev`).

### Cách 3 — `dp[i]` lưu hẳn chuỗi (không cần `prev`)

**Giải thích.** Thay vì lưu độ dài rồi dựng lại, ta lưu **chính chuỗi tốt nhất kết thúc tại `nums[i]`**: `chain[i]` là list. Với mỗi `i`: tìm `j < i` chia được `nums[i]` có `len(chain[j])` lớn nhất (`best_j`), rồi `chain[i] = chain[best_j] + [nums[i]]` (list mới), hoặc `[nums[i]]` nếu không có `j`. Đáp án là chuỗi dài nhất trong các `chain[i]`. Dễ hiểu và dễ viết hơn, nhưng tốn bộ nhớ.

In [14]:
def largest_divisible_subset_lists(nums):
    nums = sorted(nums)
    chain = []                                       # chain[i]: chuỗi tốt nhất kết thúc tại nums[i]
    for i, x in enumerate(nums):
        best_prefix = []                             # chuỗi tốt nhất trong các chuỗi kết thúc tại số nhỏ hơn chia được x
        for j in range(i):
            if x % nums[j] == 0 and len(chain[j]) > len(best_prefix):
                best_prefix = chain[j]
        chain.append(best_prefix + [x])              # list mới, không sửa chain[j]
    return max(chain, key=len)


for nums, best_len in examples:
    res = largest_divisible_subset_lists(nums)
    print(res, "độ dài", len(res), "(kỳ vọng", best_len, ")")

[1, 2] độ dài 2 (kỳ vọng 2 )
[1, 2, 4, 8] độ dài 4 (kỳ vọng 4 )


In [15]:
# Code nộp LeetCode cho Cách 3
class SolutionLists:
    def largestDivisibleSubset(self, nums: List[int]) -> List[int]:
        nums = sorted(nums)
        chain = []
        for i, x in enumerate(nums):
            best_prefix = []
            for j in range(i):
                if x % nums[j] == 0 and len(chain[j]) > len(best_prefix):
                    best_prefix = chain[j]
            chain.append(best_prefix + [x])
        return max(chain, key=len)


print(SolutionLists().largestDivisibleSubset([3, 4, 8, 16]))

[4, 8, 16]


**Độ phức tạp Cách 3.** **Time `O(n²)`** cộng chi phí sao chép list: mỗi `chain[i]` có độ dài tối đa `L` (độ dài chuỗi dài nhất; với giá trị `<= 2·10^9` thì `L <= 31`, vì mỗi bước nhân ít nhất `2`), nên sao chép tốn `O(L)` mỗi `i` ⇒ tổng `O(n² + n·L)`. **Space `O(n·L)`** (mỗi `i` giữ một chuỗi). Với bài này `L <= 31` nên chi phí thêm nhỏ.

### Cách 4 — Đệ quy có nhớ (nhìn về phía số lớn hơn)

**Giải thích.** Đổi hướng: `best(i)` = **chuỗi dài nhất bắt đầu** tại `nums[i]` (sắp xếp tăng dần), gồm `nums[i]` và chuỗi tốt nhất bắt đầu tại một số **lớn hơn** `nums[k]` (`k > i`) với `nums[k] % nums[i] == 0`. Trả về `(độ_dài, kế_tiếp)` để dựng lại. Đáp án là `best(i)` lớn nhất qua mọi `i`. `lru_cache` đảm bảo mỗi `i` được tính một lần. (Độ sâu đệ quy tối đa bằng độ dài chuỗi `<= 31`, nên không lo `RecursionError`.)

In [16]:
def largest_divisible_subset_memo(nums):
    nums = sorted(nums)
    n = len(nums)

    @lru_cache(maxsize=None)
    def best(i):
        """(độ dài chuỗi dài nhất bắt đầu tại i, chỉ số kế tiếp hoặc -1)."""
        length, nxt = 1, -1
        for k in range(i + 1, n):
            if nums[k] % nums[i] == 0:
                l2 = best(k)[0] + 1
                if l2 > length:
                    length, nxt = l2, k
        return length, nxt

    start = max(range(n), key=lambda i: best(i)[0])     # điểm bắt đầu của chuỗi dài nhất
    ans = []
    k = start
    while k != -1:
        ans.append(nums[k])
        k = best(k)[1]
    return ans


for nums, best_len in examples:
    res = largest_divisible_subset_memo(nums)
    print(res, "độ dài", len(res), "(kỳ vọng", best_len, ")")

[1, 2] độ dài 2 (kỳ vọng 2 )
[1, 2, 4, 8] độ dài 4 (kỳ vọng 4 )


In [17]:
# Code nộp LeetCode cho Cách 4
class SolutionMemo:
    def largestDivisibleSubset(self, nums: List[int]) -> List[int]:
        nums.sort()
        n = len(nums)

        @lru_cache(maxsize=None)
        def best(i):
            length, nxt = 1, -1
            for k in range(i + 1, n):
                if nums[k] % nums[i] == 0:
                    l2 = best(k)[0] + 1
                    if l2 > length:
                        length, nxt = l2, k
            return length, nxt

        start = max(range(n), key=lambda i: best(i)[0])
        ans = []
        while start != -1:
            ans.append(nums[start])
            start = best(start)[1]
        return ans


print(SolutionMemo().largestDivisibleSubset([1, 2, 3, 6, 12]))

[1, 2, 6, 12]


**Độ phức tạp Cách 4.** **Time `O(n²)`** (`n` trạng thái, mỗi trạng thái duyệt `O(n)` số lớn hơn), **Space `O(n)`** (bộ nhớ đệm + ngăn xếp đệ quy `<= 31` tầng).

## 8. TRACE từng bước

Trace **Cách 2** trên `nums = [9, 1, 8, 4, 72, 2]` (chưa sắp xếp!). Sau khi sắp xếp: `[1, 2, 4, 8, 9, 72]`. Ta in từng `(i, j)` thỏa chia hết, cập nhật `dp`/`prev`, rồi dựng lại.

In [18]:
def trace_dp(nums):
    nums = sorted(nums)
    print("đã sắp xếp:", nums)
    n = len(nums)
    dp, prev = [1] * n, [-1] * n
    best = 0
    for i in range(n):
        print(f"--- i={i}, x={nums[i]}")
        for j in range(i):
            divisible = nums[i] % nums[j] == 0
            if divisible and dp[j] + 1 > dp[i]:
                dp[i], prev[i] = dp[j] + 1, j
                print(f"    j={j} ({nums[j]}): {nums[i]} % {nums[j]} == 0, dp[{j}]+1 = {dp[j] + 1} > cũ -> dp[{i}] = {dp[i]}, prev[{i}] = {j}")
            elif divisible:
                print(f"    j={j} ({nums[j]}): chia hết nhưng dp[{j}]+1 = {dp[j] + 1} không tốt hơn dp[{i}] = {dp[i]}")
            else:
                print(f"    j={j} ({nums[j]}): {nums[i]} % {nums[j]} = {nums[i] % nums[j]} != 0 -> bỏ")
        if dp[i] > dp[best]:
            best = i
        print(f"    kết quả: dp = {dp}, prev = {prev}, best = {best}")
    ans, k = [], best
    while k != -1:
        print(f"dựng lại: lấy nums[{k}] = {nums[k]}, prev = {prev[k]}")
        ans.append(nums[k])
        k = prev[k]
    return ans[::-1]


res = trace_dp([9, 1, 8, 4, 72, 2])
print("đáp án:", res)
assert res == [1, 2, 4, 8, 72]

đã sắp xếp: [1, 2, 4, 8, 9, 72]
--- i=0, x=1
    kết quả: dp = [1, 1, 1, 1, 1, 1], prev = [-1, -1, -1, -1, -1, -1], best = 0
--- i=1, x=2
    j=0 (1): 2 % 1 == 0, dp[0]+1 = 2 > cũ -> dp[1] = 2, prev[1] = 0
    kết quả: dp = [1, 2, 1, 1, 1, 1], prev = [-1, 0, -1, -1, -1, -1], best = 1
--- i=2, x=4
    j=0 (1): 4 % 1 == 0, dp[0]+1 = 2 > cũ -> dp[2] = 2, prev[2] = 0
    j=1 (2): 4 % 2 == 0, dp[1]+1 = 3 > cũ -> dp[2] = 3, prev[2] = 1
    kết quả: dp = [1, 2, 3, 1, 1, 1], prev = [-1, 0, 1, -1, -1, -1], best = 2
--- i=3, x=8
    j=0 (1): 8 % 1 == 0, dp[0]+1 = 2 > cũ -> dp[3] = 2, prev[3] = 0
    j=1 (2): 8 % 2 == 0, dp[1]+1 = 3 > cũ -> dp[3] = 3, prev[3] = 1
    j=2 (4): 8 % 4 == 0, dp[2]+1 = 4 > cũ -> dp[3] = 4, prev[3] = 2
    kết quả: dp = [1, 2, 3, 4, 1, 1], prev = [-1, 0, 1, 2, -1, -1], best = 3
--- i=4, x=9
    j=0 (1): 9 % 1 == 0, dp[0]+1 = 2 > cũ -> dp[4] = 2, prev[4] = 0
    j=1 (2): 9 % 2 = 1 != 0 -> bỏ
    j=2 (4): 9 % 4 = 1 != 0 -> bỏ
    j=3 (8): 9 % 8 = 1 != 0 -> bỏ
    kết quả

Bảng trace Cách 2 với `[1, 2, 4, 8, 9, 72]` (đã sắp xếp):

| `i` | `x` | `j` chia được `x` | `dp[j]` | `dp[i]` | `prev[i]` |
|---|---|---|---|---|---|
| 0 | 1 | – | – | 1 | −1 |
| 1 | 2 | `0 (1)` | 1 | 2 | 0 |
| 2 | 4 | `0 (1)`, `1 (2)` | 1, 2 | 3 | 1 |
| 3 | 8 | `0, 1, 2` | 1, 2, 3 | 4 | 2 |
| 4 | 9 | `0 (1)` | 1 | 2 | 0 |
| 5 | 72 | `0, 1, 2, 3, 4` | 1, 2, 3, 4, 2 | 5 | 3 |

Nút thắt thú vị ở `72`: nó chia hết cho `9` (`dp = 2`) **và** cho `8` (`dp = 4`); ta chọn `j = 3` (số `8`) vì `dp[3]` lớn nhất. Dựng lại: `best = 5` ⇒ `72`; `prev = 3` ⇒ `8`; `prev = 2` ⇒ `4`; `prev = 1` ⇒ `2`; `prev = 0` ⇒ `1`; `prev = −1` dừng. Đảo: `[1, 2, 4, 8, 72]`.

Bây giờ trace **Cách 4** (đệ quy có nhớ) trên `[1, 2, 3, 6, 12]` để thấy `best(i)` được tính từ phía lớn về nhỏ.

In [19]:
def trace_memo(nums):
    nums = sorted(nums)
    n = len(nums)
    cache = {}

    def best(i, depth=0):
        pad = "  " * depth
        if i in cache:
            print(f"{pad}best({i}) = {cache[i]} (đã nhớ)")
            return cache[i]
        print(f"{pad}tính best({i}) cho x = {nums[i]}")
        length, nxt = 1, -1
        for k in range(i + 1, n):
            if nums[k] % nums[i] == 0:
                l2 = best(k, depth + 1)[0] + 1
                if l2 > length:
                    length, nxt = l2, k
        cache[i] = (length, nxt)
        print(f"{pad}=> best({i}) = {cache[i]}")
        return cache[i]

    best(0)
    start = max(range(n), key=lambda i: best(i)[0])
    ans, k = [], start
    while k != -1:
        ans.append(nums[k])
        k = cache[k][1]
    return ans


print("đáp án:", trace_memo([1, 2, 3, 6, 12]))

tính best(0) cho x = 1
  tính best(1) cho x = 2
    tính best(3) cho x = 6
      tính best(4) cho x = 12
      => best(4) = (1, -1)
    => best(3) = (2, 4)
    best(4) = (1, -1) (đã nhớ)
  => best(1) = (3, 3)
  tính best(2) cho x = 3
    best(3) = (2, 4) (đã nhớ)
    best(4) = (1, -1) (đã nhớ)
  => best(2) = (3, 3)
  best(3) = (2, 4) (đã nhớ)
  best(4) = (1, -1) (đã nhớ)
=> best(0) = (4, 1)
best(0) = (4, 1) (đã nhớ)
best(1) = (3, 3) (đã nhớ)
best(2) = (3, 3) (đã nhớ)
best(3) = (2, 4) (đã nhớ)
best(4) = (1, -1) (đã nhớ)
đáp án: [1, 2, 6, 12]


## 9. Code nộp LeetCode

Bản nộp dùng **Cách 2** (sắp xếp + DP + `prev`). Lưu ý `nums.sort()` sắp xếp **tại chỗ** (được phép vì đề chỉ yêu cầu trả tập con).

In [20]:
from typing import List


class Solution:
    def largestDivisibleSubset(self, nums: List[int]) -> List[int]:
        nums.sort()                                          # tăng dần: số lớn chia số nhỏ
        n = len(nums)
        dp = [1] * n                                         # dp[i]: chuỗi dài nhất kết thúc tại nums[i]
        prev = [-1] * n                                      # prev[i]: phần tử đứng trước nums[i] trong chuỗi
        best = 0                                             # chỉ số kết thúc của chuỗi dài nhất
        for i in range(n):
            for j in range(i):
                if nums[i] % nums[j] == 0 and dp[j] + 1 > dp[i]:
                    dp[i] = dp[j] + 1
                    prev[i] = j
            if dp[i] > dp[best]:
                best = i
        ans = []
        while best != -1:                                    # dựng lại bằng prev
            ans.append(nums[best])
            best = prev[best]
        return ans[::-1]


sol = Solution()
print(sol.largestDivisibleSubset([1, 2, 3]))
print(sol.largestDivisibleSubset([1, 2, 4, 8]))

[1, 2]
[1, 2, 4, 8]


## 10. Vì sao thuật toán đúng

**(1) Tập hợp lệ ⇔ chuỗi chia hết.** Với tập `S` đã sắp xếp tăng dần `a₁ < a₂ < ... < a_k`: nếu `S` hợp lệ thì mỗi cặp liền kề chia hết (cặp `(a_i, a_{i+1})` có số lớn chia số nhỏ vì số nhỏ không chia được số lớn). Ngược lại, nếu mỗi `a_i | a_{i+1}` thì bắc cầu cho `a_i | a_j` với mọi `i < j`; nên `S` hợp lệ.

**(2) Công thức `dp`.** `dp[i]` là độ dài chuỗi chia hết dài nhất **kết thúc** tại `nums[i]` (đã sắp xếp). Một chuỗi như vậy là `nums[i]` đứng sau một chuỗi kết thúc tại `nums[j]` nào đó (`j < i`, `nums[j] | nums[i]`) hoặc đứng một mình. Với mỗi `j` cố định, chuỗi dài nhất kết thúc tại `nums[j]` có độ dài `dp[j]` và **kéo dài hợp lệ** bằng `nums[i]` (liền kề `nums[j] | nums[i]`; các cặp khác của chuỗi cũ vẫn hợp lệ). Lấy `max` trên mọi `j` ⇒ `dp[i] = 1 + max(...)` ✓ (cấu trúc con tối ưu: phần đầu của chuỗi dài nhất kết thúc tại `i` cũng phải là chuỗi dài nhất kết thúc tại `j`, nếu không ta thay bằng chuỗi dài hơn).

**(3) Đáp án.** Chuỗi chia hết dài nhất toàn cục kết thúc ở **một** phần tử nào đó, nên nó có độ dài `max(dp)`.

**(4) Dựng lại.** `prev[i]` chỉ đến `j` đã đạt `dp[i]` (lần cập nhật cuối cùng với `dp[j] + 1 > dp[i]` — tức `j` đầu tiên đạt giá trị lớn nhất). Đi ngược `prev` cho một chuỗi các phần tử có `nums[prev] | nums[k]` ở mỗi bước, với độ dài `dp[best]`; mọi cặp hợp lệ nhờ (1).

**(5) Cách 3 và Cách 4** cùng ý: Cách 3 lưu chuỗi thay vì độ dài; Cách 4 đảo hướng (chuỗi **bắt đầu** tại `i`, mở rộng về phía số lớn hơn) và cũng có cấu trúc con tối ưu tương tự.

**Nhiều đáp án:** khi nhiều `j` cho cùng `dp[j]` lớn nhất, ta chọn `j` đầu tiên; đề cho phép trả **một** đáp án bất kỳ, vì vậy test phải dùng **hàm kiểm tra tính hợp lệ** + so độ dài với oracle.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`, `L` là độ dài chuỗi dài nhất (với giá trị `<= 2·10^9` thì `L <= 31`).

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn tập con | `O(2^n · n²)` | `O(n)` |
| 2. Sắp xếp + DP + `prev` | `O(n²)` | `O(n)` |
| 3. DP lưu chuỗi | `O(n² + n·L)` | `O(n·L)` |
| 4. Đệ quy có nhớ | `O(n²)` | `O(n)` |

Với `n = 1000`: `~500 000` phép chia lấy dư — rất nhẹ (vài chục mili-giây trong Python).

## 12. Bộ test

Bài này **có nhiều đáp án đúng**, nên ta kiểm tra bằng hàm `is_valid_answer(nums, ans)` (mọi phần tử thuộc `nums`, không lặp, mọi cặp chia hết) **và** so sánh **độ dài** `len(ans)` với độ dài tối ưu từ một **oracle** (vét cạn khi `n` nhỏ; ngược lại một DP theo giá trị viết độc lập).

In [21]:
sol = Solution()
impls = {
    "brute": largest_divisible_subset_brute,
    "dp": largest_divisible_subset_dp,
    "lists": largest_divisible_subset_lists,
    "memo": largest_divisible_subset_memo,
    "lc_dp": lambda nums: SolutionDP().largestDivisibleSubset(list(nums)),
    "lc_lists": lambda nums: SolutionLists().largestDivisibleSubset(list(nums)),
    "lc_memo": lambda nums: SolutionMemo().largestDivisibleSubset(list(nums)),
    "leetcode": lambda nums: sol.largestDivisibleSubset(list(nums)),    # list(nums): LeetCode sắp xếp tại chỗ, ta giữ bản gốc
}


def is_valid_answer(nums, ans):
    """ans phải là tập con (không lặp) của nums mà mọi cặp chia hết lẫn nhau."""
    if len(set(ans)) != len(ans):
        return False
    if not set(ans) <= set(nums):
        return False
    return is_divisible_set(ans)


def oracle_length(nums):
    """DP theo giá trị, viết độc lập (dict thay cho mảng chỉ số)."""
    best = {}
    for x in sorted(nums):
        best[x] = 1 + max((best[d] for d in best if x % d == 0), default=0)
    return max(best.values())


def check_all(nums, label, expected_len=None, brute_ok=True):
    target_len = oracle_length(nums)
    if expected_len is not None:
        assert target_len == expected_len, (label, target_len, expected_len)
    for key, fn in impls.items():
        if key == "brute" and (not brute_ok or len(nums) > 14):
            continue
        ans = fn(list(nums))
        assert is_valid_answer(nums, ans), (label, key, ans)
        assert len(ans) == target_len, (label, key, ans, target_len)
    print("PASS", label, "(độ dài tối ưu", target_len, ")")


for idx, (nums, best_len) in enumerate(examples, 1):
    check_all(nums, f"ví dụ {idx}", expected_len=best_len)

PASS ví dụ 1 (độ dài tối ưu 2 )
PASS ví dụ 2 (độ dài tối ưu 4 )


### 12.1. Các trường hợp biên

Một phần tử; chứa số `1`; toàn số nguyên tố (độ dài `1`); chuỗi lũy thừa; hai chuỗi tranh nhau; số rất lớn; mảng chưa sắp xếp.

In [22]:
check_all([1], "một phần tử [1]", 1)
check_all([7], "một phần tử [7]", 1)
check_all([2, 3], "hai số nguyên tố", 1)
check_all([2, 3, 5, 7, 11, 13], "toàn số nguyên tố -> độ dài 1", 1)
check_all([5, 1], "hai số, có 1", 2)
check_all([8, 4, 2, 1], "chưa sắp xếp, chuỗi lũy thừa của 2", 4)
check_all([3, 4, 16, 8], "ví dụ [3,4,16,8] -> [4,8,16]", 3)
check_all([4, 8, 10, 240], "[4,8,10,240] -> [4,8,240]", 3)
check_all([1, 2, 3, 6, 12], "hai đáp án cùng độ dài 4", 4)
check_all([9, 1, 8, 4, 72, 2], "ví dụ trace -> [1,2,4,8,72]", 5)
check_all([1, 2, 4, 5, 20], "nhánh 4 và 5 tranh nhau trước 20", 4)
check_all([2, 3, 4, 9, 8, 27, 16, 81], "hai họ lũy thừa 2 và 3", 4)
big = 2 * 10**9
check_all([1, big], "số lớn nhất theo đề", 2)
check_all([big, big // 2, big // 4, big // 8, 1], "chuỗi chia đôi từ 2e9", 5)
powers = [2 ** k for k in range(31)]               # 1, 2, ..., 2^30 <= 2e9
check_all(powers, "31 lũy thừa của 2 (độ dài tối đa có thể)", 31)
check_all([6, 10, 15, 30], "6,10,15 chéo nhau, 30 chia hết cả ba -> độ dài 2", 2)

PASS một phần tử [1] (độ dài tối ưu 1 )
PASS một phần tử [7] (độ dài tối ưu 1 )
PASS hai số nguyên tố (độ dài tối ưu 1 )
PASS toàn số nguyên tố -> độ dài 1 (độ dài tối ưu 1 )
PASS hai số, có 1 (độ dài tối ưu 2 )
PASS chưa sắp xếp, chuỗi lũy thừa của 2 (độ dài tối ưu 4 )
PASS ví dụ [3,4,16,8] -> [4,8,16] (độ dài tối ưu 3 )
PASS [4,8,10,240] -> [4,8,240] (độ dài tối ưu 3 )
PASS hai đáp án cùng độ dài 4 (độ dài tối ưu 4 )
PASS ví dụ trace -> [1,2,4,8,72] (độ dài tối ưu 5 )
PASS nhánh 4 và 5 tranh nhau trước 20 (độ dài tối ưu 4 )
PASS hai họ lũy thừa 2 và 3 (độ dài tối ưu 4 )
PASS số lớn nhất theo đề (độ dài tối ưu 2 )
PASS chuỗi chia đôi từ 2e9 (độ dài tối ưu 5 )
PASS 31 lũy thừa của 2 (độ dài tối đa có thể) (độ dài tối ưu 31 )
PASS 6,10,15 chéo nhau, 30 chia hết cả ba -> độ dài 2 (độ dài tối ưu 2 )


### 12.2. Phản ví dụ cho các cách làm sai

Hai lỗi phổ biến: (a) **quên sắp xếp**; (b) **tham lam** (đi từ nhỏ đến lớn, thấy chia hết thì lấy).

In [23]:
def wrong_no_sort(nums):
    n = len(nums)
    dp, prev, best = [1] * n, [-1] * n, 0
    for i in range(n):
        for j in range(i):
            if nums[i] % nums[j] == 0 and dp[j] + 1 > dp[i]:      # SAI: chưa sắp xếp -> số lớn có thể đứng trước
                dp[i], prev[i] = dp[j] + 1, j
        if dp[i] > dp[best]:
            best = i
    ans = []
    while best != -1:
        ans.append(nums[best]); best = prev[best]
    return ans[::-1]


def wrong_greedy(nums):
    nums = sorted(nums)
    chain = [nums[0]]
    for x in nums[1:]:
        if x % chain[-1] == 0:                                    # SAI: tham lam, bị kẹt ở số nhỏ đầu tiên
            chain.append(x)
    return chain


print("quên sort:", wrong_no_sort([8, 4, 2, 1]), "(độ dài tối ưu 4)")
print("tham lam :", wrong_greedy([3, 4, 8, 16]), "(độ dài tối ưu 3)")
assert len(wrong_no_sort([8, 4, 2, 1])) < 4
assert len(wrong_greedy([3, 4, 8, 16])) < 3

quên sort: [8] (độ dài tối ưu 4)
tham lam : [3] (độ dài tối ưu 3)


### 12.3. Đối chiếu ngẫu nhiên (có seed) — vét cạn và oracle

(1) `n` nhỏ (`1..11`), giá trị nhỏ (nhiều quan hệ chia hết) **so với vét cạn** (có sắp xếp độc lập). (2) `n` lớn (tới `1000`), nhiều dạng: giá trị nhỏ (`1..2000`), bội của một số, lũy thừa trộn nhiễu, giá trị rất lớn (`<= 2·10^9`), **so với oracle theo giá trị**. Mọi test đều sinh **số khác nhau từng đôi** đúng đề.

In [24]:
rng = random.Random(368)
for trial in range(600):                                    # nhỏ: có vét cạn
    n = rng.randint(1, 11)
    hi = rng.choice([12, 30, 60, 120])
    nums = rng.sample(range(1, hi + 1), min(n, hi))
    rng.shuffle(nums)
    best_brute = largest_divisible_subset_brute(sorted(nums))
    for key, fn in impls.items():
        if key == "brute":
            continue                                        # vét cạn đã tính ở trên (best_brute)
        ans = fn(list(nums))
        assert is_valid_answer(nums, ans), (key, nums, ans)
        assert len(ans) == len(best_brute), (key, nums, ans, best_brute)
print("PASS 600 mảng nhỏ ngẫu nhiên đối chiếu với vét cạn")

PASS 600 mảng nhỏ ngẫu nhiên đối chiếu với vét cạn


In [25]:
def gen_big(rng, kind):
    n = rng.randint(1, 1000)
    if kind == "small_range":
        pool = range(1, 2001)
        return rng.sample(pool, min(n, 2000))
    if kind == "multiples":
        base = rng.randint(1, 50)
        pool = {base * rng.randint(1, 3000) for _ in range(n)}
        return list(pool)
    if kind == "powers_noise":
        pool = {2 ** rng.randint(0, 30) for _ in range(50)} | {3 ** rng.randint(0, 19) for _ in range(30)}
        while len(pool) < n:
            pool.add(rng.randint(1, 2 * 10**9))
        return list(pool)[:n] if len(pool) > n else list(pool)
    pool = {rng.randint(1, 2 * 10**9) for _ in range(n)}          # "huge": ngẫu nhiên thưa
    return list(pool)


rng = random.Random(3680)
for trial in range(120):
    kind = rng.choice(["small_range", "multiples", "powers_noise", "huge"])
    nums = gen_big(rng, kind)
    rng.shuffle(nums)
    assert 1 <= len(nums) <= 1000 and len(set(nums)) == len(nums) and all(1 <= v <= 2 * 10**9 for v in nums)
    ref = oracle_length(nums)
    for key, fn in impls.items():
        if key == "brute":
            continue
        ans = fn(list(nums))
        assert is_valid_answer(nums, ans), (key, kind)
        assert len(ans) == ref, (key, kind, len(ans), ref)
print("PASS 120 mảng lớn ngẫu nhiên (tới 1000 phần tử) đối chiếu với oracle theo giá trị")

# Thời gian cho n = 1000, giá trị toàn bội của 1..1000 (nhiều quan hệ chia hết nhất)
nums = list(range(1, 1001))
start = time.perf_counter()
ans = sol.largestDivisibleSubset(list(nums))
print(f"n=1000 (1..1000): độ dài {len(ans)}, {time.perf_counter() - start:.3f}s", ans)
assert is_valid_answer(nums, ans) and len(ans) == oracle_length(nums) == 10      # 1,2,4,...,512

PASS 120 mảng lớn ngẫu nhiên (tới 1000 phần tử) đối chiếu với oracle theo giá trị
n=1000 (1..1000): độ dài 10, 0.018s [1, 2, 4, 8, 16, 32, 64, 128, 256, 512]


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên sắp xếp

In [26]:
print("sai:", wrong_no_sort([8, 4, 2, 1]), "| đúng:", Solution().largestDivisibleSubset([8, 4, 2, 1]))

sai: [8] | đúng: [1, 2, 4, 8]


**Vì sao sai:** với `[8, 4, 2, 1]` chưa sắp xếp, `nums[i] % nums[j]` với `j < i` là `4 % 8`, `2 % 8`, ... đều khác `0`, nên không cặp nào được nối và kết quả chỉ có `1` phần tử. DP cần **số nhỏ đứng trước số lớn**. **Cách sửa:** `nums.sort()` ngay đầu.

### Lỗi 2: Tham lam (lấy số nhỏ nhất rồi nhặt các bội)

In [27]:
print("sai:", wrong_greedy([3, 4, 8, 16]), "| đúng:", Solution().largestDivisibleSubset([3, 4, 8, 16]))

sai: [3] | đúng: [4, 8, 16]


**Vì sao sai:** tham lam bắt đầu bằng `3` rồi không nhặt được số nào (`4, 8, 16` đều không chia hết cho `3`), trong khi chuỗi tốt nhất `[4, 8, 16]` **không** chứa `3`. Lựa chọn đầu tiên ảnh hưởng toàn bộ phần sau — cần quy hoạch động thử mọi điểm kết thúc. **Cách sửa:** DP `dp[i]` như trên.

### Lỗi 3: Dựng lại đáp án không kiểm tra chia hết

In [28]:
def sai_dung_lai(nums):
    nums = sorted(nums)
    n = len(nums)
    dp = [1] * n
    for i in range(n):
        for j in range(i):
            if nums[i] % nums[j] == 0:
                dp[i] = max(dp[i], dp[j] + 1)
    m = max(dp)
    ans = []
    for i in range(n - 1, -1, -1):                 # SAI: chỉ lấy theo dp == m, m-1, ... mà bỏ kiểm tra chia hết
        if dp[i] == m:
            ans.append(nums[i])
            m -= 1
    return ans[::-1]


res = sai_dung_lai([2, 3, 4])
print("sai:", res, "| hợp lệ?", is_divisible_set(res), "| đúng:", Solution().largestDivisibleSubset([2, 3, 4]))
assert not is_divisible_set(res)

sai: [3, 4] | hợp lệ? False | đúng: [2, 4]


**Vì sao sai:** `dp = [1, 1, 2]`; dựng lại lấy `4` (`dp = 2`) rồi tới `3` (`dp = 1`), nhưng `3` **không** chia hết `4` ⇒ tập `[3, 4]` **không hợp lệ**. Chỉ biết độ dài chưa đủ để dựng lại — cần biết **ai đứng trước ai**. **Cách sửa:** dùng mảng `prev`, hoặc khi đi ngược kiểm tra `nums[k] % nums[i] == 0 and dp[i] == m`.

### Lỗi 4: Quên `max` khi cập nhật `dp` (ghi đè bằng `j` cuối cùng)

In [29]:
def sai_quen_max(nums):
    nums = sorted(nums)
    n = len(nums)
    dp = [1] * n
    for i in range(n):
        for j in range(i):
            if nums[i] % nums[j] == 0:
                dp[i] = dp[j] + 1                  # SAI: ghi đè bằng j cuối cùng chia hết, không lấy tốt nhất
    return max(dp)


print("sai:", sai_quen_max([1, 2, 4, 5, 20]), "| đúng:", len(Solution().largestDivisibleSubset([1, 2, 4, 5, 20])))

sai: 3 | đúng: 4


**Vì sao sai:** với `20`, các số chia được là `1, 2, 4, 5` (`dp = 1, 2, 3, 2`). Ghi đè theo thứ tự `j` làm `dp[20] = dp[5] + 1 = 3` (vì `5` là `j` cuối), trong khi tốt nhất là `dp[4] + 1 = 4`. **Cách sửa:** `dp[i] = max(dp[i], dp[j] + 1)` (hoặc so sánh `dp[j] + 1 > dp[i]`).

### Lỗi 5: Trả về độ dài thay vì tập con

In [30]:
def sai_tra_do_dai(nums):
    nums = sorted(nums)
    n = len(nums)
    dp = [1] * n
    for i in range(n):
        for j in range(i):
            if nums[i] % nums[j] == 0:
                dp[i] = max(dp[i], dp[j] + 1)
    return max(dp)                                  # SAI: đề yêu cầu trả về TẬP CON, không phải độ dài


out = sai_tra_do_dai([1, 2, 3])
print("sai:", out, type(out).__name__, "| đúng:", Solution().largestDivisibleSubset([1, 2, 3]))

sai: 2 int | đúng: [1, 2]


**Vì sao sai:** đề yêu cầu `List[int]` (một tập con). Trả về `int` bị chấm sai dù độ dài đúng. **Cách sửa:** thêm `prev` và dựng lại (mục 7, Cách 2) hoặc dùng Cách 3 (lưu chuỗi).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "**Tập con lớn nhất mà mọi cặp thỏa quan hệ có tính bắc cầu**" (chia hết, `<=` ...) ⇒ **sắp xếp** rồi quy về **chuỗi** (kiểu **LIS**): `dp[i] = 1 + max(dp[j])` với `j < i` thỏa quan hệ.
- Tính **bắc cầu** là chìa khóa để chỉ kiểm tra cặp liền kề; hãy tự hỏi "quan hệ này có bắc cầu không?" (chia hết: có; "hơn kém đúng 1": không).
- Cần **trả về chính tập con**: ghi `prev[i]` và dựng lại; hoặc lưu hẳn chuỗi nếu bộ nhớ cho phép.
- Giá trị `<= 2·10^9` ⇒ chuỗi chia hết **không quá 31 phần tử** (mỗi bước ít nhất nhân đôi): hữu ích khi phân tích bộ nhớ của Cách 3.
- Nhiều đáp án ⇒ viết **hàm kiểm tra tính hợp lệ** + so độ dài với oracle.
- Với mảng `nums` chuyển vào: nhớ `sort()` thay đổi list gốc (tùy ngữ cảnh dùng `sorted`).

## 15. Biến thể của bài

1. **Chỉ cần độ dài** (không dựng lại): `max(dp)`.
2. **Đếm số tập con lớn nhất** (như **Number of Longest Increasing Subsequence 673**): thêm `cnt[i]`.
3. **Longest Increasing Subsequence (300)**, **Russian Doll Envelopes (354)**, **Longest String Chain (1048)**: cùng khung "chuỗi dài nhất trên thứ tự".
4. **Giá trị nhỏ (`<= 10^6`)**: có thể duyệt **ước** hoặc **bội** của mỗi số theo bảng giá trị thay vì `O(n²)`.
5. **Tập con có phần tử bắt buộc** (ví dụ phải chứa `nums[0]`).
6. **Cùng khung với đồ thị:** chuỗi chia hết dài nhất = **đường đi dài nhất trong DAG** (cạnh `a → b` nếu `a | b`).

In [31]:
def lds_length(nums):
    nums = sorted(nums)
    dp = [1] * len(nums)
    for i in range(len(nums)):
        for j in range(i):
            if nums[i] % nums[j] == 0:
                dp[i] = max(dp[i], dp[j] + 1)
    return max(dp)


def count_largest_subsets(nums):
    """Số tập con chia hết có kích thước lớn nhất (mỗi tập = một chuỗi duy nhất khi sắp xếp)."""
    nums = sorted(nums)
    n = len(nums)
    dp, cnt = [1] * n, [1] * n
    for i in range(n):
        for j in range(i):
            if nums[i] % nums[j] == 0:
                if dp[j] + 1 > dp[i]:
                    dp[i], cnt[i] = dp[j] + 1, cnt[j]
                elif dp[j] + 1 == dp[i]:
                    cnt[i] += cnt[j]
    m = max(dp)
    return sum(c for d, c in zip(dp, cnt) if d == m)


def count_brute(nums):
    n = len(nums)
    best, count = 0, 0
    for mask in range(1, 1 << n):
        sub = [nums[i] for i in range(n) if mask >> i & 1]
        if is_divisible_set(sub):
            if len(sub) > best:
                best, count = len(sub), 1
            elif len(sub) == best:
                count += 1
    return count


print(lds_length([1, 2, 3, 6, 12]), count_largest_subsets([1, 2, 3, 6, 12]))     # 4 2: hai tập [1,2,6,12] và [1,3,6,12]
rng = random.Random(3681)
for _ in range(300):
    nums = rng.sample(range(1, 61), rng.randint(1, 11))
    assert count_largest_subsets(nums) == count_brute(nums), nums
    assert lds_length(nums) == len(Solution().largestDivisibleSubset(list(nums)))
print("đếm số tập con lớn nhất khớp vét cạn")

4 2
đếm số tập con lớn nhất khớp vét cạn


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn tập con | thử mọi tập con, kiểm tra mọi cặp | `O(2^n · n²)` | `O(n)` | oracle cho `n` nhỏ |
| 2. Sắp xếp + DP + `prev` | LIS trên quan hệ chia hết | `O(n²)` | `O(n)` | **nên nộp** |
| 3. DP lưu chuỗi | `chain[i] = chain[j] + [x]` | `O(n² + n·L)` | `O(n·L)` | dễ viết, không cần dựng lại |
| 4. Đệ quy có nhớ | chuỗi bắt đầu tại `i`, nhìn về số lớn | `O(n²)` | `O(n)` | quen top-down |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `12 % 4`, `4 % 12`; (b) `sorted([8, 2, 4])`; (c) với `nums = [2, 4]` đã sắp xếp, `dp` cuối cùng; (d) độ dài của `Solution().largestDivisibleSubset([1, 2, 3])`; (e) `is_divisible_set([2, 6, 5])`.

**Bài 2 (tính tay).** Với `nums = [5, 10, 2, 20, 40, 3]`: sắp xếp, lập bảng `dp`/`prev` (như mục 4) và chỉ ra **một** chuỗi dài nhất.

**Bài 3 (điền TODO).** Hoàn thành `lds_todo(nums)` (trả tập con) theo Cách 2.

In [32]:
def lds_todo(nums):
    # TODO 1: sắp xếp nums tăng dần (dùng sorted để không sửa đầu vào)
    # TODO 2: tạo dp = [1] * n và prev = [-1] * n; best = 0
    # TODO 3: hai vòng lặp i, j < i: nếu nums[i] % nums[j] == 0 và dp[j] + 1 > dp[i] thì cập nhật dp[i], prev[i]
    # TODO 4: cập nhật best nếu dp[i] > dp[best]
    # TODO 5: dựng lại bằng prev từ best, nhớ đảo ngược danh sách
    return None

# KHÔNG gọi hàm TODO ở đây để Run All không lỗi (TODO chưa điền).

**Bài 4 (tự viết lại).** Viết `lds_lists2(nums)` theo Cách 3 (mỗi `chain[i]` lưu hẳn chuỗi tốt nhất kết thúc tại `nums[i]`) **mà không nhìn lại** code ở mục 7; kiểm tra với `[3, 4, 16, 8]` (độ dài `3`).

**Bài 5 (biến thể / mini challenge).** (a) Viết `must_contain(nums, v)`: tập con chia hết **lớn nhất bắt buộc chứa phần tử `v`** (gợi ý: các phần tử còn lại phải thuộc ước hoặc bội của `v` và tạo chuỗi; có thể chạy DP trên cả mảng rồi ép đi qua `v`). (b) Viết `count_largest(nums)` đếm số tập con lớn nhất và kiểm tra `[1, 2, 3, 6, 12] → 2`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `0` và `4`; `[2, 4, 8]`; `dp = [1, 2]`; `2`; `False` (`2 | 6` nhưng `5` không chia hết 2 hay 6). Bài 3: bắt đầu bằng `nums = sorted(nums)`.
- **Mức 2 (vừa):** Bài 2: sắp xếp `[2, 3, 5, 10, 20, 40]`; `dp = [1, 1, 1, 2, 3, 4]`; `prev` của `10` là `0` (số `2`) hoặc `2` (số `5`), `20` lấy `10`, `40` lấy `20`. Bài 4: `chain.append(best_prefix + [x])`.
- **Mức 3 (sâu):** Bài 5(a): chia mảng thành các số **chia hết `v`** hoặc **là ước của `v`**; chuỗi dài nhất chứa `v` = chuỗi dài nhất kết thúc ở `v` (dp trái) + chuỗi dài nhất bắt đầu ở `v` (dp phải) − 1. 5(b): thêm `cnt[i]` như mục 15.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [33]:
# Bài 1
assert 12 % 4 == 0 and 4 % 12 == 4
assert sorted([8, 2, 4]) == [2, 4, 8]
dp = [1, 1]
for i in range(2):
    for j in range(i):
        if [2, 4][i] % [2, 4][j] == 0:
            dp[i] = max(dp[i], dp[j] + 1)
assert dp == [1, 2]
assert len(Solution().largestDivisibleSubset([1, 2, 3])) == 2
assert is_divisible_set([2, 6, 5]) is False

# Bài 2
nums2 = [5, 10, 2, 20, 40, 3]
res2 = Solution().largestDivisibleSubset(list(nums2))
assert is_valid_answer(nums2, res2) and len(res2) == 4 and res2[-2:] == [20, 40]


# Bài 3
def lds_todo_ans(nums):
    nums = sorted(nums)
    n = len(nums)
    dp, prev, best = [1] * n, [-1] * n, 0
    for i in range(n):
        for j in range(i):
            if nums[i] % nums[j] == 0 and dp[j] + 1 > dp[i]:
                dp[i], prev[i] = dp[j] + 1, j
        if dp[i] > dp[best]:
            best = i
    ans = []
    while best != -1:
        ans.append(nums[best])
        best = prev[best]
    return ans[::-1]


for nums, best_len in examples:
    ans = lds_todo_ans(nums)
    assert is_valid_answer(nums, ans) and len(ans) == best_len


# Bài 4
def lds_lists2(nums):
    nums = sorted(nums)
    chain = []
    for i, x in enumerate(nums):
        best_prefix = []
        for j in range(i):
            if x % nums[j] == 0 and len(chain[j]) > len(best_prefix):
                best_prefix = chain[j]
        chain.append(best_prefix + [x])
    return max(chain, key=len)


ans4 = lds_lists2([3, 4, 16, 8])
assert ans4 == [4, 8, 16]
rng = random.Random(3682)
for _ in range(300):
    nums = rng.sample(range(1, 200), rng.randint(1, 40))
    a = lds_lists2(nums)
    assert is_valid_answer(nums, a) and len(a) == oracle_length(nums)


# Bài 5(b)
def count_largest(nums):
    nums = sorted(nums)
    n = len(nums)
    dp, cnt = [1] * n, [1] * n
    for i in range(n):
        for j in range(i):
            if nums[i] % nums[j] == 0:
                if dp[j] + 1 > dp[i]:
                    dp[i], cnt[i] = dp[j] + 1, cnt[j]
                elif dp[j] + 1 == dp[i]:
                    cnt[i] += cnt[j]
    m = max(dp)
    return sum(c for d, c in zip(dp, cnt) if d == m)


assert count_largest([1, 2, 3, 6, 12]) == 2
assert count_largest([1, 2, 4, 8]) == 1
assert count_largest([2, 3, 5]) == 3


# Bài 5(a): tập con chia hết lớn nhất bắt buộc chứa v
def must_contain(nums, v):
    nums = sorted(nums)
    n = len(nums)
    iv = nums.index(v)
    left = [1] * n                         # chuỗi dài nhất kết thúc tại i
    right = [1] * n                        # chuỗi dài nhất bắt đầu tại i
    for i in range(n):
        for j in range(i):
            if nums[i] % nums[j] == 0:
                left[i] = max(left[i], left[j] + 1)
    for i in range(n - 1, -1, -1):
        for k in range(i + 1, n):
            if nums[k] % nums[i] == 0:
                right[i] = max(right[i], right[k] + 1)
    return left[iv] + right[iv] - 1


def must_contain_brute(nums, v):
    n = len(nums)
    best = 0
    for mask in range(1, 1 << n):
        sub = [nums[i] for i in range(n) if mask >> i & 1]
        if v in sub and is_divisible_set(sub):
            best = max(best, len(sub))
    return best


assert must_contain([1, 2, 3, 6, 12], 3) == 4
assert must_contain([3, 4, 8, 16], 3) == 1
for _ in range(200):
    nums = rng.sample(range(1, 61), rng.randint(1, 10))
    v = rng.choice(nums)
    assert must_contain(nums, v) == must_contain_brute(nums, v), (nums, v)
print("Tất cả đáp án đều PASS")

Tất cả đáp án đều PASS


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (sắp xếp + DP + `prev`)

```python
class Solution:
    def largestDivisibleSubset(self, nums: List[int]) -> List[int]:
        nums.sort()
        n = len(nums)
        dp = [1] * n
        prev = [-1] * n
        best = 0
        for i in range(n):
            for j in range(i):
                if nums[i] % nums[j] == 0 and dp[j] + 1 > dp[i]:
                    dp[i] = dp[j] + 1
                    prev[i] = j
            if dp[i] > dp[best]:
                best = i
        ans = []
        while best != -1:
            ans.append(nums[best])
            best = prev[best]
        return ans[::-1]
```

**Checklist**

- [ ] Đã **sắp xếp** tăng dần trước khi DP (để `nums[i] % nums[j]` với `j < i` là "lớn chia nhỏ")?
- [ ] `dp[i]` khởi tạo `1`, cập nhật bằng **`max`** trên mọi `j` chia được (không ghi đè bằng `j` cuối)?
- [ ] Có `prev` (hoặc lưu chuỗi) để **dựng lại** tập con, và trả về **list** chứ không phải độ dài?
- [ ] Hiểu vì sao chỉ cần kiểm tra cặp liền kề (tính **bắc cầu** của chia hết)?
- [ ] Test bằng **hàm kiểm tra tính hợp lệ** (nhiều đáp án đúng) và so độ dài với vét cạn/oracle?
- [ ] Đã thử: một phần tử, toàn số nguyên tố, có số `1`, chuỗi lũy thừa của `2`, số lớn `2·10^9`, `n = 1000`?